# 02 — Manda 3-Date HPF Fusion: Q1-Ready Evaluation

This notebook creates the operational 3 m fused products and Q1-oriented fusion validation.

## Spatial rule used for Manda

- PlanetScope is not clipped with the Manda administrative AOI.
- The prepared Planet union grid defines the operational spatial extent.
- Sentinel-2 is warped, aligned and clipped to that Planet grid.
- Fused pixels are written only where the relevant Planet date and Sentinel-2 both contain valid data.
- Areas missing from the Planet download remain NoData.
- Classification later uses the common valid footprint across all required dates.

## Fusion pairs

- Planet 25 January 2026 + Sentinel-2 23 January 2026
- Planet 5 March 2026 + Sentinel-2 4 March 2026
- Planet 7 April 2026 + Sentinel-2 10 April 2026

The 22 April Planet image is not fused because no sufficiently clear Sentinel-2 image is used for that date.

## Products

- Operational fused products: 3 m Planet grid
- Fused NDVI: derived after fusion
- Full-resolution evaluation
- Wald-style reduced-resolution validation
- Q1 publication metric tables
- Two classification-stream manifests


> Corrected pair: PlanetScope **5 March 2026** + Sentinel-2 **4 March 2026**.

## VS Code local-PC version

এই notebook Google Colab বা local PC ব্যবহার করে না।

মূল project path:

```text
D:\Boro Rice Classification
```

Notebook চালানোর আগে `setup_windows.bat` চালিয়ে `Python (Boro Rice Project)` kernel নির্বাচন করুন।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:
# CELL 1 — Portable project path for local use and GitHub reproduction

from pathlib import Path
import os

# Recommended: set BORO_PROJECT_ROOT to the local project directory.
# If it is not set, launch Jupyter from the repository root.
PROJECT_ROOT = Path(
    os.environ.get("BORO_PROJECT_ROOT", str(Path.cwd()))
).expanduser().resolve()

DATA_ROOT = PROJECT_ROOT / "Data"
OUTPUTS_ROOT = PROJECT_ROOT / "Outputs"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data directory was not found: {DATA_ROOT}\n"
        "Set BORO_PROJECT_ROOT or launch Jupyter from the repository root."
    )

OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Outputs root:", OUTPUTS_ROOT)


In [ ]:
# CELL 2 — Import installed local packages

from pathlib import Path
import math
import json
import warnings

import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.warp import reproject
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message=".*not georeferenced.*",
)

print("Rasterio:", rasterio.__version__)

In [ ]:
# CELL 3 — Local paths and fusion pairs

STUDY_AREA = "Manda"

PLANET_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Prepared_Planet"
)

PLANET_NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Planet_NDVI"
)

S2_DIR = (
    DATA_ROOT
    / STUDY_AREA
    / "Sentinel2"
)

FUSION_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion"
)

FUSION_NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion_NDVI"
)

CLASSIFICATION_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Classification_Inputs"
)

REPORT_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion_Reports"
)

RR_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion_Reduced_Resolution"
)

for folder in [
    FUSION_DIR,
    FUSION_NDVI_DIR,
    CLASSIFICATION_DIR,
    REPORT_DIR,
    RR_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

PAIRS = {
    "Jan": {
        "planet": (
            PLANET_DIR
            / "Manda_25_jan_SR_prepared_full.tif"
        ),
        "s2": (
            S2_DIR
            / "S2_Manda_20260123_SR.tif"
        ),
        "output_label": "25_jan",
        "planet_date": "20260125",
        "s2_date": "20260123",
    },
    "Mar": {
        "planet": (
            PLANET_DIR
            / "Manda_5_march_SR_prepared_full.tif"
        ),
        "s2": (
            S2_DIR
            / "S2_Manda_20260304_SR.tif"
        ),
        "output_label": "5_march",
        "planet_date": "20260305",
        "s2_date": "20260304",
    },
    "Apr1": {
        "planet": (
            PLANET_DIR
            / "Manda_7_april_SR_prepared_full.tif"
        ),
        "s2": (
            S2_DIR
            / "S2_Manda_20260410_SR.tif"
        ),
        "output_label": "7_april",
        "planet_date": "20260407",
        "s2_date": "20260410",
    },
}

# Planet band order: Blue, Green, Red, NIR.
PLANET_BANDS = [1, 2, 3, 4]

# Sentinel-2 file must also be Blue, Green, Red, NIR.
S2_BANDS = [1, 2, 3, 4]

NODATA = -9999.0
BLOCK_SIZE = 512

HPF_GAIN = 1.0
REFLECTANCE_MIN = 0.0
REFLECTANCE_MAX = 1.2

# Metric settings
METRIC_DATA_RANGE = 1.0
RR_SCALE_FACTOR = 3
SSIM_TILE_SIZE = 96
SSIM_MIN_VALID_FRACTION = 0.95
RANDOM_SEED = 42

print("Planet input:", PLANET_DIR)
print("Sentinel-2 input:", S2_DIR)
print("Fusion output:", FUSION_DIR)
print("Fusion reports:", REPORT_DIR)


In [ ]:
# CELL 4 — Helper functions

def iter_windows(width, height, block_size=512):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=min(block_size, width - col_off),
                height=min(block_size, height - row_off),
            )


def expanded_window(window, width, height, halo):
    col0 = max(0, int(window.col_off) - halo)
    row0 = max(0, int(window.row_off) - halo)
    col1 = min(
        width,
        int(window.col_off + window.width) + halo,
    )
    row1 = min(
        height,
        int(window.row_off + window.height) + halo,
    )

    expanded = Window(
        col0,
        row0,
        col1 - col0,
        row1 - row0,
    )

    core_row = int(window.row_off) - row0
    core_col = int(window.col_off) - col0

    core = (
        slice(
            core_row,
            core_row + int(window.height),
        ),
        slice(
            core_col,
            core_col + int(window.width),
        ),
    )

    return expanded, core


def valid_mask(data, nodata):
    return (
        np.all(np.isfinite(data), axis=0)
        & np.all(data != nodata, axis=0)
        & np.any(data > -100, axis=0)
    )


def normalized_lowpass(array, valid, sigma):
    valid_float = valid.astype("float32")

    weighted = gaussian_filter(
        np.where(valid, array, 0.0).astype("float32"),
        sigma=sigma,
        mode="nearest",
    )

    weights = gaussian_filter(
        valid_float,
        sigma=sigma,
        mode="nearest",
    )

    result = np.full(
        array.shape,
        np.nan,
        dtype="float32",
    )

    np.divide(
        weighted,
        weights,
        out=result,
        where=weights > 1e-6,
    )

    return result


def output_profile(reference, count):
    profile = reference.profile.copy()

    profile.update(
        driver="GTiff",
        count=count,
        dtype="float32",
        nodata=NODATA,
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        blockxsize=BLOCK_SIZE,
        blockysize=BLOCK_SIZE,
        BIGTIFF="IF_SAFER",
    )

    return profile


def safe_correlation(
    sum_x,
    sum_y,
    sum_x2,
    sum_y2,
    sum_xy,
    n,
):
    if n <= 1:
        return np.nan

    mean_x = sum_x / n
    mean_y = sum_y / n

    variance_x = sum_x2 / n - mean_x ** 2
    variance_y = sum_y2 / n - mean_y ** 2
    covariance = sum_xy / n - mean_x * mean_y

    if variance_x <= 0 or variance_y <= 0:
        return np.nan

    return covariance / math.sqrt(
        variance_x * variance_y
    )


In [ ]:
# CELL 5 — Verify six input files

input_rows = []

for stage, pair in PAIRS.items():
    for sensor, path in [
        ("Planet", pair["planet"]),
        ("Sentinel2", pair["s2"]),
    ]:
        if not path.exists():
            raise FileNotFoundError(
                f"{stage} {sensor} file পাওয়া যায়নি: {path}"
            )

        with rasterio.open(path) as src:
            if src.count != 4:
                raise ValueError(
                    f"{path.name}: expected 4 bands, got {src.count}"
                )

            if src.crs is None:
                raise ValueError(
                    f"{path.name}: CRS missing."
                )

            # Read one representative valid block.
            valid_found = False

            for _, window in src.block_windows(1):
                data = src.read(window=window).astype("float32")
                nodata = (
                    src.nodata
                    if src.nodata is not None
                    else NODATA
                )

                if np.any(valid_mask(data, nodata)):
                    valid_found = True
                    break

            if not valid_found:
                raise ValueError(
                    f"{path.name}: zero valid pixels."
                )

            input_rows.append({
                "stage": stage,
                "sensor": sensor,
                "path": str(path),
                "bands": src.count,
                "width": src.width,
                "height": src.height,
                "resolution": str(src.res),
                "crs": str(src.crs),
                "valid_data_found": valid_found,
            })

input_df = pd.DataFrame(input_rows)
display(input_df)

print("✅ All fusion inputs are ready.")


In [ ]:
# CELL 6 — HPF fusion function

def fuse_one_pair(stage, pair):
    planet_path = pair["planet"]
    s2_path = pair["s2"]
    label = pair["output_label"]

    fused_path = (
        FUSION_DIR
        / f"Fused_Manda_{label}.tif"
    )

    fused_ndvi_path = (
        FUSION_NDVI_DIR
        / f"NDVI_Fused_Manda_{label}.tif"
    )

    with rasterio.open(planet_path) as planet:
        with rasterio.open(s2_path) as s2:
            planet_nodata = (
                planet.nodata
                if planet.nodata is not None
                else NODATA
            )

            s2_nodata = (
                s2.nodata
                if s2.nodata is not None
                else NODATA
            )

            planet_resolution = abs(
                float(planet.transform.a)
            )

            s2_resolution = abs(
                float(s2.transform.a)
            )

            sigma = float(
                np.clip(
                    (s2_resolution / planet_resolution) / 2.355,
                    0.8,
                    3.0,
                )
            )

            halo = max(
                4,
                int(math.ceil(4.0 * sigma)),
            )

            total_blocks = (
                math.ceil(planet.width / BLOCK_SIZE)
                * math.ceil(planet.height / BLOCK_SIZE)
            )

            band_stats = [
                {
                    "n": 0,
                    "sum_s": 0.0,
                    "sum_f": 0.0,
                    "sum_s2": 0.0,
                    "sum_f2": 0.0,
                    "sum_sf": 0.0,
                    "sum_sq_error": 0.0,
                    "sum_abs_error": 0.0,
                }
                for _ in range(4)
            ]

            total_valid_pixels = 0
            sam_sum = 0.0
            sam_count = 0

            with WarpedVRT(
                s2,
                crs=planet.crs,
                transform=planet.transform,
                width=planet.width,
                height=planet.height,
                resampling=Resampling.bilinear,
                src_nodata=s2_nodata,
                nodata=NODATA,
                dtype="float32",
            ) as s2_vrt:
                with rasterio.open(
                    fused_path,
                    "w",
                    **output_profile(planet, 4),
                ) as fused_dst:
                    with rasterio.open(
                        fused_ndvi_path,
                        "w",
                        **output_profile(planet, 1),
                    ) as ndvi_dst:
                        descriptions = [
                            "Blue_Fused_Reflectance",
                            "Green_Fused_Reflectance",
                            "Red_Fused_Reflectance",
                            "NIR_Fused_Reflectance",
                        ]

                        for index, description in enumerate(
                            descriptions,
                            start=1,
                        ):
                            fused_dst.set_band_description(
                                index,
                                description,
                            )

                        ndvi_dst.set_band_description(
                            1,
                            "NDVI_Fused",
                        )

                        fused_dst.update_tags(
                            study_area="Manda",
                            fusion_method="HPF",
                            planet_date=pair["planet_date"],
                            sentinel2_date=pair["s2_date"],
                            hpf_sigma=sigma,
                            hpf_gain=HPF_GAIN,
                        )

                        progress = tqdm(
                            total=total_blocks,
                            desc=f"Fusing {stage}",
                        )

                        for window in iter_windows(
                            planet.width,
                            planet.height,
                            BLOCK_SIZE,
                        ):
                            expanded, core = expanded_window(
                                window,
                                planet.width,
                                planet.height,
                                halo,
                            )

                            planet_data = planet.read(
                                window=expanded,
                            ).astype("float32")

                            s2_data = s2_vrt.read(
                                window=expanded,
                            ).astype("float32")

                            planet_valid = valid_mask(
                                planet_data,
                                planet_nodata,
                            )

                            s2_valid = valid_mask(
                                s2_data,
                                NODATA,
                            )

                            overlap = (
                                planet_valid
                                & s2_valid
                            )

                            fused_expanded = np.full(
                                planet_data.shape,
                                NODATA,
                                dtype="float32",
                            )

                            for band in range(4):
                                lowpass = normalized_lowpass(
                                    planet_data[band],
                                    overlap,
                                    sigma,
                                )

                                high_frequency = (
                                    planet_data[band]
                                    - lowpass
                                )

                                fused_values = (
                                    s2_data[band]
                                    + HPF_GAIN
                                    * high_frequency
                                )

                                fused_values = np.clip(
                                    fused_values,
                                    REFLECTANCE_MIN,
                                    REFLECTANCE_MAX,
                                )

                                fused_expanded[
                                    band,
                                    overlap,
                                ] = fused_values[overlap]

                            fused_core = fused_expanded[
                                :,
                                core[0],
                                core[1],
                            ]

                            overlap_core = overlap[core]

                            s2_core = s2_data[
                                :,
                                core[0],
                                core[1],
                            ]

                            fused_dst.write(
                                fused_core,
                                window=window,
                            )

                            red = fused_core[2]
                            nir = fused_core[3]
                            denominator = nir + red

                            ndvi_valid = (
                                overlap_core
                                & np.isfinite(denominator)
                                & (np.abs(denominator) > 1e-8)
                            )

                            ndvi = np.full(
                                red.shape,
                                NODATA,
                                dtype="float32",
                            )

                            ndvi[ndvi_valid] = np.clip(
                                (
                                    nir[ndvi_valid]
                                    - red[ndvi_valid]
                                )
                                / denominator[ndvi_valid],
                                -1.0,
                                1.0,
                            )

                            ndvi_dst.write(
                                ndvi,
                                1,
                                window=window,
                            )

                            total_valid_pixels += int(
                                overlap_core.sum()
                            )

                            if np.any(overlap_core):
                                for band in range(4):
                                    x = s2_core[band][
                                        overlap_core
                                    ].astype("float64")

                                    y = fused_core[band][
                                        overlap_core
                                    ].astype("float64")

                                    stat = band_stats[band]
                                    n = x.size

                                    stat["n"] += int(n)
                                    stat["sum_s"] += float(x.sum())
                                    stat["sum_f"] += float(y.sum())
                                    stat["sum_s2"] += float(
                                        np.sum(x ** 2)
                                    )
                                    stat["sum_f2"] += float(
                                        np.sum(y ** 2)
                                    )
                                    stat["sum_sf"] += float(
                                        np.sum(x * y)
                                    )

                                    error = y - x

                                    stat["sum_sq_error"] += float(
                                        np.sum(error ** 2)
                                    )
                                    stat["sum_abs_error"] += float(
                                        np.sum(np.abs(error))
                                    )

                                x_vectors = s2_core[
                                    :,
                                    overlap_core,
                                ].astype("float64")

                                y_vectors = fused_core[
                                    :,
                                    overlap_core,
                                ].astype("float64")

                                dot = np.sum(
                                    x_vectors * y_vectors,
                                    axis=0,
                                )

                                norm_x = np.linalg.norm(
                                    x_vectors,
                                    axis=0,
                                )

                                norm_y = np.linalg.norm(
                                    y_vectors,
                                    axis=0,
                                )

                                sam_valid = (
                                    (norm_x > 1e-12)
                                    & (norm_y > 1e-12)
                                )

                                if np.any(sam_valid):
                                    cosine = np.clip(
                                        dot[sam_valid]
                                        / (
                                            norm_x[sam_valid]
                                            * norm_y[sam_valid]
                                        ),
                                        -1.0,
                                        1.0,
                                    )

                                    angles = np.arccos(cosine)

                                    sam_sum += float(angles.sum())
                                    sam_count += int(angles.size)

                            progress.update(1)

                        progress.close()

            if total_valid_pixels == 0:
                raise ValueError(
                    f"{stage}: Planet and S2 have zero valid overlap."
                )

            metric_rows = []
            band_names = ["Blue", "Green", "Red", "NIR"]

            for band_name, stat in zip(
                band_names,
                band_stats,
            ):
                n = stat["n"]

                rmse = math.sqrt(
                    stat["sum_sq_error"] / n
                )

                mae = (
                    stat["sum_abs_error"] / n
                )

                correlation = safe_correlation(
                    stat["sum_s"],
                    stat["sum_f"],
                    stat["sum_s2"],
                    stat["sum_f2"],
                    stat["sum_sf"],
                    n,
                )

                metric_rows.append({
                    "stage": stage,
                    "band": band_name,
                    "rmse_vs_s2": rmse,
                    "mae_vs_s2": mae,
                    "correlation_vs_s2": correlation,
                    "valid_pixels": n,
                })

            summary = {
                "stage": stage,
                "planet_date": pair["planet_date"],
                "s2_date": pair["s2_date"],
                "sigma": sigma,
                "valid_pixels": total_valid_pixels,
                "valid_percent_of_planet_grid": (
                    100.0
                    * total_valid_pixels
                    / (planet.width * planet.height)
                ),
                "mean_sam_degrees": (
                    math.degrees(sam_sum / sam_count)
                    if sam_count
                    else np.nan
                ),
                "fused_path": str(fused_path),
                "fused_ndvi_path": str(fused_ndvi_path),
            }

    return summary, metric_rows


In [ ]:
# CELL 7 — Run the three fusions

summary_rows = []
metric_rows = []

for stage in ["Jan", "Mar", "Apr1"]:
    print()
    print("=" * 70)
    print("RUNNING:", stage)
    print("=" * 70)

    summary, metrics = fuse_one_pair(
        stage,
        PAIRS[stage],
    )

    summary_rows.append(summary)
    metric_rows.extend(metrics)

    print("✅", summary["fused_path"])
    print("✅", summary["fused_ndvi_path"])

fusion_summary = pd.DataFrame(summary_rows)
fusion_metrics_matrix = pd.DataFrame(metric_rows)

display(fusion_summary)
display(fusion_metrics_matrix)

fusion_summary.to_csv(
    CLASSIFICATION_DIR
    / "Manda_Fusion_Summary.csv",
    index=False,
)

fusion_metrics_matrix.to_csv(
    CLASSIFICATION_DIR
    / "Manda_Fusion_Metrics_Matrix.csv",
    index=False,
)

print("✅ Three-date fusion complete.")


In [ ]:
# CELL 8 — Create the two classification streams

apr22_planet = (
    PLANET_DIR
    / "Manda_22_april_SR_prepared_full.tif"
)

apr22_ndvi = (
    PLANET_NDVI_DIR
    / "Manda_22_april_NDVI_full.tif"
)

if not apr22_planet.exists():
    raise FileNotFoundError(apr22_planet)

if not apr22_ndvi.exists():
    raise FileNotFoundError(apr22_ndvi)

four_band_rows = [
    {
        "date_key": "Jan",
        "input_path": str(
            FUSION_DIR
            / "Fused_Manda_25_jan.tif"
        ),
        "input_type": "Fused 4-band",
    },
    {
        "date_key": "Mar",
        "input_path": str(
            FUSION_DIR
            / "Fused_Manda_5_march.tif"
        ),
        "input_type": "Fused 4-band",
    },
    {
        "date_key": "Apr1",
        "input_path": str(
            FUSION_DIR
            / "Fused_Manda_7_april.tif"
        ),
        "input_type": "Fused 4-band",
    },
    {
        "date_key": "Apr2",
        "input_path": str(apr22_planet),
        "input_type": "Prepared Planet 4-band",
    },
]

ndvi_rows = [
    {
        "date_key": "Jan",
        "input_path": str(
            FUSION_NDVI_DIR
            / "NDVI_Fused_Manda_25_jan.tif"
        ),
        "input_type": "NDVI from fused image",
    },
    {
        "date_key": "Mar",
        "input_path": str(
            FUSION_NDVI_DIR
            / "NDVI_Fused_Manda_5_march.tif"
        ),
        "input_type": "NDVI from fused image",
    },
    {
        "date_key": "Apr1",
        "input_path": str(
            FUSION_NDVI_DIR
            / "NDVI_Fused_Manda_7_april.tif"
        ),
        "input_type": "NDVI from fused image",
    },
    {
        "date_key": "Apr2",
        "input_path": str(apr22_ndvi),
        "input_type": "NDVI from prepared Planet",
    },
]

four_band_manifest = pd.DataFrame(four_band_rows)
ndvi_manifest = pd.DataFrame(ndvi_rows)

# Verify existence, band count and common grid.
for stream_name, manifest, expected_bands in [
    ("4-band", four_band_manifest, 4),
    ("NDVI", ndvi_manifest, 1),
]:
    reference_grid = None

    for _, row in manifest.iterrows():
        path = Path(row["input_path"])

        if not path.exists():
            raise FileNotFoundError(path)

        with rasterio.open(path) as src:
            if src.count != expected_bands:
                raise ValueError(
                    f"{path.name}: expected {expected_bands} bands."
                )

            grid = (
                str(src.crs),
                src.transform,
                src.width,
                src.height,
            )

            if reference_grid is None:
                reference_grid = grid
            elif grid != reference_grid:
                raise ValueError(
                    f"{stream_name} stream grids do not match."
                )

four_band_manifest.to_csv(
    CLASSIFICATION_DIR
    / "Manda_4Band_Classification_Manifest.csv",
    index=False,
)

ndvi_manifest.to_csv(
    CLASSIFICATION_DIR
    / "Manda_NDVI_Classification_Manifest.csv",
    index=False,
)

print("STREAM A — 4-band classification")
display(four_band_manifest)

print("STREAM B — NDVI classification")
display(ndvi_manifest)

print("✅ Both classification streams are ready.")
print("Classification folder:", CLASSIFICATION_DIR)


In [ ]:
# CELL 9 — Q1 metric helper functions

def scalar_statistics(reference, estimate):
    reference = np.asarray(reference, dtype="float64")
    estimate = np.asarray(estimate, dtype="float64")

    valid = (
        np.isfinite(reference)
        & np.isfinite(estimate)
    )

    reference = reference[valid]
    estimate = estimate[valid]

    if reference.size < 2:
        return {
            "n": int(reference.size),
            "rmse": np.nan,
            "mae": np.nan,
            "psnr_db": np.nan,
            "cc": np.nan,
            "uiqi": np.nan,
        }

    error = estimate - reference
    rmse = float(np.sqrt(np.mean(error ** 2)))
    mae = float(np.mean(np.abs(error)))

    if rmse == 0:
        psnr_db = np.inf
    else:
        psnr_db = float(
            20.0 * np.log10(METRIC_DATA_RANGE / rmse)
        )

    reference_mean = float(np.mean(reference))
    estimate_mean = float(np.mean(estimate))

    reference_var = float(np.var(reference))
    estimate_var = float(np.var(estimate))

    covariance = float(
        np.mean(
            (reference - reference_mean)
            * (estimate - estimate_mean)
        )
    )

    if reference_var > 0 and estimate_var > 0:
        cc = float(
            covariance
            / np.sqrt(reference_var * estimate_var)
        )
    else:
        cc = np.nan

    uiqi_denominator = (
        (reference_var + estimate_var)
        * (
            reference_mean ** 2
            + estimate_mean ** 2
        )
    )

    if abs(uiqi_denominator) > 1e-15:
        uiqi = float(
            4.0
            * covariance
            * reference_mean
            * estimate_mean
            / uiqi_denominator
        )
    else:
        uiqi = np.nan

    return {
        "n": int(reference.size),
        "rmse": rmse,
        "mae": mae,
        "psnr_db": psnr_db,
        "cc": cc,
        "uiqi": uiqi,
    }


def weighted_window_ssim(
    reference,
    estimate,
    valid_mask_array,
    tile_size=96,
    min_valid_fraction=0.95,
):
    reference = np.asarray(reference, dtype="float32")
    estimate = np.asarray(estimate, dtype="float32")
    valid_mask_array = np.asarray(
        valid_mask_array,
        dtype=bool,
    )

    height, width = reference.shape
    weighted_sum = 0.0
    weight_total = 0
    accepted_tiles = 0

    for row0 in range(0, height, tile_size):
        for col0 in range(0, width, tile_size):
            row1 = min(height, row0 + tile_size)
            col1 = min(width, col0 + tile_size)

            ref_tile = reference[row0:row1, col0:col1]
            est_tile = estimate[row0:row1, col0:col1]
            valid_tile = valid_mask_array[
                row0:row1,
                col0:col1,
            ]

            if min(ref_tile.shape) < 7:
                continue

            valid_fraction = float(valid_tile.mean())

            if valid_fraction < min_valid_fraction:
                continue

            ref_valid = ref_tile[valid_tile]
            est_valid = est_tile[valid_tile]

            if ref_valid.size < 49:
                continue

            # Only the small invalid remainder is filled.
            ref_fill = float(np.mean(ref_valid))
            est_fill = float(np.mean(est_valid))

            ref_work = np.where(
                valid_tile,
                ref_tile,
                ref_fill,
            )

            est_work = np.where(
                valid_tile,
                est_tile,
                est_fill,
            )

            score = structural_similarity(
                ref_work,
                est_work,
                data_range=METRIC_DATA_RANGE,
                win_size=7,
                gaussian_weights=True,
                sigma=1.5,
                use_sample_covariance=False,
            )

            weight = int(valid_tile.sum())
            weighted_sum += float(score) * weight
            weight_total += weight
            accepted_tiles += 1

    return {
        "ssim": (
            weighted_sum / weight_total
            if weight_total > 0
            else np.nan
        ),
        "ssim_valid_pixels": weight_total,
        "ssim_tiles": accepted_tiles,
    }


def spectral_angle_mean_degrees(
    reference_cube,
    estimate_cube,
    valid_mask_array,
):
    reference_vectors = reference_cube[
        :,
        valid_mask_array,
    ].astype("float64")

    estimate_vectors = estimate_cube[
        :,
        valid_mask_array,
    ].astype("float64")

    dot_product = np.sum(
        reference_vectors * estimate_vectors,
        axis=0,
    )

    reference_norm = np.linalg.norm(
        reference_vectors,
        axis=0,
    )

    estimate_norm = np.linalg.norm(
        estimate_vectors,
        axis=0,
    )

    valid = (
        (reference_norm > 1e-12)
        & (estimate_norm > 1e-12)
    )

    if not np.any(valid):
        return np.nan

    cosine = np.clip(
        dot_product[valid]
        / (
            reference_norm[valid]
            * estimate_norm[valid]
        ),
        -1.0,
        1.0,
    )

    return float(
        np.degrees(
            np.mean(np.arccos(cosine))
        )
    )


def ergas_value(
    per_band_rmse,
    per_band_reference_mean,
    resolution_ratio,
):
    ratios = []

    for rmse, mean_value in zip(
        per_band_rmse,
        per_band_reference_mean,
    ):
        if (
            np.isfinite(rmse)
            and np.isfinite(mean_value)
            and abs(mean_value) > 1e-12
        ):
            ratios.append(
                (rmse / mean_value) ** 2
            )

    if not ratios:
        return np.nan

    return float(
        (100.0 / resolution_ratio)
        * np.sqrt(np.mean(ratios))
    )


def high_frequency_correlation(
    reference,
    estimate,
    valid_mask_array,
    sigma,
):
    reference_low = normalized_lowpass(
        reference,
        valid_mask_array,
        sigma,
    )

    estimate_low = normalized_lowpass(
        estimate,
        valid_mask_array,
        sigma,
    )

    reference_hf = reference - reference_low
    estimate_hf = estimate - estimate_low

    valid = (
        valid_mask_array
        & np.isfinite(reference_hf)
        & np.isfinite(estimate_hf)
    )

    if valid.sum() < 2:
        return np.nan

    return float(
        np.corrcoef(
            reference_hf[valid],
            estimate_hf[valid],
        )[0, 1]
    )


def evaluate_cube(
    reference_cube,
    estimate_cube,
    spatial_reference_cube,
    valid_mask_array,
    stage,
    product,
    resolution_ratio,
    hf_sigma,
):
    band_names = ["Blue", "Green", "Red", "NIR"]
    rows = []
    rmse_values = []
    reference_means = []

    for band_index, band_name in enumerate(band_names):
        reference_band = reference_cube[band_index]
        estimate_band = estimate_cube[band_index]

        statistics = scalar_statistics(
            reference_band[valid_mask_array],
            estimate_band[valid_mask_array],
        )

        ssim_result = weighted_window_ssim(
            reference_band,
            estimate_band,
            valid_mask_array,
            tile_size=SSIM_TILE_SIZE,
            min_valid_fraction=SSIM_MIN_VALID_FRACTION,
        )

        hf_cc = high_frequency_correlation(
            spatial_reference_cube[band_index],
            estimate_band,
            valid_mask_array,
            sigma=hf_sigma,
        )

        reference_mean = float(
            np.mean(
                reference_band[valid_mask_array]
            )
        )

        rmse_values.append(statistics["rmse"])
        reference_means.append(reference_mean)

        rows.append({
            "stage": stage,
            "product": product,
            "band": band_name,
            "valid_pixels": statistics["n"],
            "rmse": statistics["rmse"],
            "mae": statistics["mae"],
            "psnr_db": statistics["psnr_db"],
            "ssim": ssim_result["ssim"],
            "ssim_tiles": ssim_result["ssim_tiles"],
            "cc": statistics["cc"],
            "uiqi": statistics["uiqi"],
            "spatial_hf_cc": hf_cc,
            "reference_mean": reference_mean,
        })

    sam_degrees = spectral_angle_mean_degrees(
        reference_cube,
        estimate_cube,
        valid_mask_array,
    )

    ergas = ergas_value(
        rmse_values,
        reference_means,
        resolution_ratio=resolution_ratio,
    )

    summary = {
        "stage": stage,
        "product": product,
        "valid_pixels": int(valid_mask_array.sum()),
        "mean_rmse": float(np.nanmean(rmse_values)),
        "mean_mae": float(
            np.nanmean(
                [row["mae"] for row in rows]
            )
        ),
        "mean_psnr_db": float(
            np.nanmean(
                [row["psnr_db"] for row in rows]
            )
        ),
        "mean_ssim": float(
            np.nanmean(
                [row["ssim"] for row in rows]
            )
        ),
        "mean_cc": float(
            np.nanmean(
                [row["cc"] for row in rows]
            )
        ),
        "mean_uiqi": float(
            np.nanmean(
                [row["uiqi"] for row in rows]
            )
        ),
        "mean_spatial_hf_cc": float(
            np.nanmean(
                [row["spatial_hf_cc"] for row in rows]
            )
        ),
        "sam_degrees": sam_degrees,
        "ergas": ergas,
    }

    return rows, summary


def write_float_cube(
    output_path,
    cube,
    reference_dataset,
    nodata=NODATA,
):
    profile = reference_dataset.profile.copy()

    profile.update(
        driver="GTiff",
        count=cube.shape[0],
        dtype="float32",
        nodata=nodata,
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        BIGTIFF="IF_SAFER",
    )

    with rasterio.open(
        output_path,
        "w",
        **profile,
    ) as dst:
        dst.write(
            cube.astype("float32")
        )


In [ ]:
# CELL 10 — Full-resolution evaluation

full_resolution_band_rows = []
full_resolution_summary_rows = []

for stage, pair in PAIRS.items():
    fused_path = (
        FUSION_DIR
        / f"Fused_Manda_{pair['output_label']}.tif"
    )

    with rasterio.open(pair["planet"]) as planet:
        with rasterio.open(pair["s2"]) as s2:
            with rasterio.open(fused_path) as fused:
                planet_nodata = (
                    planet.nodata
                    if planet.nodata is not None
                    else NODATA
                )

                fused_nodata = (
                    fused.nodata
                    if fused.nodata is not None
                    else NODATA
                )

                s2_nodata = (
                    s2.nodata
                    if s2.nodata is not None
                    else NODATA
                )

                with WarpedVRT(
                    s2,
                    crs=planet.crs,
                    transform=planet.transform,
                    width=planet.width,
                    height=planet.height,
                    resampling=Resampling.bilinear,
                    src_nodata=s2_nodata,
                    nodata=NODATA,
                    dtype="float32",
                ) as s2_vrt:
                    # Publication evaluation uses a reproducible,
                    # spatially distributed sample instead of loading
                    # the full 3 m cube into memory.
                    rng = np.random.default_rng(
                        RANDOM_SEED
                    )

                    sample_reference = [
                        [] for _ in range(4)
                    ]
                    sample_fused = [
                        [] for _ in range(4)
                    ]
                    sample_planet = [
                        [] for _ in range(4)
                    ]

                    maximum_sample_per_block = 3000
                    maximum_total_sample = 300000

                    for window in iter_windows(
                        planet.width,
                        planet.height,
                        BLOCK_SIZE,
                    ):
                        planet_data = planet.read(
                            window=window,
                        ).astype("float32")

                        fused_data = fused.read(
                            window=window,
                        ).astype("float32")

                        s2_data = s2_vrt.read(
                            window=window,
                        ).astype("float32")

                        valid = (
                            valid_mask(
                                planet_data,
                                planet_nodata,
                            )
                            & valid_mask(
                                fused_data,
                                fused_nodata,
                            )
                            & valid_mask(
                                s2_data,
                                NODATA,
                            )
                        )

                        valid_indices = np.flatnonzero(
                            valid.ravel()
                        )

                        if valid_indices.size == 0:
                            continue

                        sample_size = min(
                            maximum_sample_per_block,
                            valid_indices.size,
                        )

                        selected = rng.choice(
                            valid_indices,
                            size=sample_size,
                            replace=False,
                        )

                        for band_index in range(4):
                            sample_reference[
                                band_index
                            ].append(
                                s2_data[
                                    band_index
                                ].ravel()[selected]
                            )

                            sample_fused[
                                band_index
                            ].append(
                                fused_data[
                                    band_index
                                ].ravel()[selected]
                            )

                            sample_planet[
                                band_index
                            ].append(
                                planet_data[
                                    band_index
                                ].ravel()[selected]
                            )

                    reference_sample_cube = np.vstack([
                        np.concatenate(values)[
                            :maximum_total_sample
                        ]
                        for values in sample_reference
                    ])

                    fused_sample_cube = np.vstack([
                        np.concatenate(values)[
                            :maximum_total_sample
                        ]
                        for values in sample_fused
                    ])

                    planet_sample_cube = np.vstack([
                        np.concatenate(values)[
                            :maximum_total_sample
                        ]
                        for values in sample_planet
                    ])

                    valid_sample = np.ones(
                        reference_sample_cube.shape[1],
                        dtype=bool,
                    )

                    band_names = [
                        "Blue",
                        "Green",
                        "Red",
                        "NIR",
                    ]

                    rmse_values = []
                    reference_means = []
                    cc_values = []
                    uiqi_values = []
                    psnr_values = []
                    mae_values = []
                    hf_cc_values = []

                    for band_index, band_name in enumerate(
                        band_names
                    ):
                        statistics = scalar_statistics(
                            reference_sample_cube[
                                band_index
                            ],
                            fused_sample_cube[
                                band_index
                            ],
                        )

                        spatial_statistics = scalar_statistics(
                            planet_sample_cube[
                                band_index
                            ],
                            fused_sample_cube[
                                band_index
                            ],
                        )

                        full_resolution_band_rows.append({
                            "stage": stage,
                            "band": band_name,
                            "sample_pixels": statistics["n"],
                            "rmse_vs_s2": statistics["rmse"],
                            "mae_vs_s2": statistics["mae"],
                            "psnr_db_vs_s2": statistics["psnr_db"],
                            "cc_vs_s2": statistics["cc"],
                            "uiqi_vs_s2": statistics["uiqi"],
                            "cc_vs_planet": spatial_statistics["cc"],
                        })

                        rmse_values.append(
                            statistics["rmse"]
                        )

                        mae_values.append(
                            statistics["mae"]
                        )

                        psnr_values.append(
                            statistics["psnr_db"]
                        )

                        cc_values.append(
                            statistics["cc"]
                        )

                        uiqi_values.append(
                            statistics["uiqi"]
                        )

                        hf_cc_values.append(
                            spatial_statistics["cc"]
                        )

                        reference_means.append(
                            float(
                                np.mean(
                                    reference_sample_cube[
                                        band_index
                                    ]
                                )
                            )
                        )

                    sam = spectral_angle_mean_degrees(
                        reference_sample_cube,
                        fused_sample_cube,
                        valid_sample,
                    )

                    ergas = ergas_value(
                        rmse_values,
                        reference_means,
                        resolution_ratio=(
                            abs(s2.transform.a)
                            / abs(planet.transform.a)
                        ),
                    )

                    full_resolution_summary_rows.append({
                        "stage": stage,
                        "planet_date": pair["planet_date"],
                        "sentinel2_date": pair["s2_date"],
                        "sample_pixels": int(
                            reference_sample_cube.shape[1]
                        ),
                        "mean_rmse_vs_s2": float(
                            np.nanmean(rmse_values)
                        ),
                        "mean_mae_vs_s2": float(
                            np.nanmean(mae_values)
                        ),
                        "mean_psnr_db_vs_s2": float(
                            np.nanmean(psnr_values)
                        ),
                        "mean_cc_vs_s2": float(
                            np.nanmean(cc_values)
                        ),
                        "mean_uiqi_vs_s2": float(
                            np.nanmean(uiqi_values)
                        ),
                        "mean_cc_vs_planet": float(
                            np.nanmean(hf_cc_values)
                        ),
                        "sam_degrees": sam,
                        "ergas": ergas,
                        "note": (
                            "Full-resolution sample evaluation; "
                            "SSIM is reported in Wald validation."
                        ),
                    })

full_resolution_band_df = pd.DataFrame(
    full_resolution_band_rows
)

full_resolution_summary_df = pd.DataFrame(
    full_resolution_summary_rows
)

display(full_resolution_band_df.round(6))
display(full_resolution_summary_df.round(6))

full_resolution_band_df.to_csv(
    REPORT_DIR
    / "Q1_FullResolution_PerBand_Metrics.csv",
    index=False,
)

full_resolution_summary_df.to_csv(
    REPORT_DIR
    / "Q1_FullResolution_Date_Summary.csv",
    index=False,
)

print("✅ Full-resolution evaluation complete.")


In [ ]:
# CELL 11 — Wald-style reduced-resolution validation

wald_band_rows = []
wald_summary_rows = []

for stage, pair in PAIRS.items():
    print()
    print("=" * 70)
    print("WALD VALIDATION:", stage)
    print("=" * 70)

    with rasterio.open(pair["s2"]) as s2:
        with rasterio.open(pair["planet"]) as planet:
            s2_nodata = (
                s2.nodata
                if s2.nodata is not None
                else NODATA
            )

            planet_nodata = (
                planet.nodata
                if planet.nodata is not None
                else NODATA
            )

            reference = s2.read().astype("float32")

            reference_valid = valid_mask(
                reference,
                s2_nodata,
            )

            # Planet data placed on the original 10 m S2 grid.
            with WarpedVRT(
                planet,
                crs=s2.crs,
                transform=s2.transform,
                width=s2.width,
                height=s2.height,
                resampling=Resampling.bilinear,
                src_nodata=planet_nodata,
                nodata=NODATA,
                dtype="float32",
            ) as planet_10m_vrt:
                planet_10m = planet_10m_vrt.read().astype(
                    "float32"
                )

            planet_valid = valid_mask(
                planet_10m,
                NODATA,
            )

            # Degrade S2 from 10 m to approximately 30 m.
            coarse_width = int(
                math.ceil(
                    s2.width / RR_SCALE_FACTOR
                )
            )

            coarse_height = int(
                math.ceil(
                    s2.height / RR_SCALE_FACTOR
                )
            )

            coarse_transform = (
                s2.transform
                * Affine.scale(
                    s2.width / coarse_width,
                    s2.height / coarse_height,
                )
            )

            coarse = np.full(
                (
                    4,
                    coarse_height,
                    coarse_width,
                ),
                NODATA,
                dtype="float32",
            )

            baseline = np.full(
                reference.shape,
                NODATA,
                dtype="float32",
            )

            for band_index in range(4):
                reproject(
                    source=reference[band_index],
                    destination=coarse[band_index],
                    src_transform=s2.transform,
                    src_crs=s2.crs,
                    src_nodata=s2_nodata,
                    dst_transform=coarse_transform,
                    dst_crs=s2.crs,
                    dst_nodata=NODATA,
                    resampling=Resampling.average,
                )

                reproject(
                    source=coarse[band_index],
                    destination=baseline[band_index],
                    src_transform=coarse_transform,
                    src_crs=s2.crs,
                    src_nodata=NODATA,
                    dst_transform=s2.transform,
                    dst_crs=s2.crs,
                    dst_nodata=NODATA,
                    resampling=Resampling.bilinear,
                )

            baseline_valid = valid_mask(
                baseline,
                NODATA,
            )

            common_valid = (
                reference_valid
                & planet_valid
                & baseline_valid
            )

            if common_valid.sum() == 0:
                raise ValueError(
                    f"{stage}: no valid pixels for Wald validation."
                )

            sigma_rr = float(
                np.clip(
                    RR_SCALE_FACTOR / 2.355,
                    0.8,
                    3.0,
                )
            )

            wald_fused = np.full(
                reference.shape,
                NODATA,
                dtype="float32",
            )

            for band_index in range(4):
                lowpass = normalized_lowpass(
                    planet_10m[band_index],
                    common_valid,
                    sigma=sigma_rr,
                )

                high_frequency = (
                    planet_10m[band_index]
                    - lowpass
                )

                values = (
                    baseline[band_index]
                    + HPF_GAIN * high_frequency
                )

                values = np.clip(
                    values,
                    REFLECTANCE_MIN,
                    REFLECTANCE_MAX,
                )

                wald_fused[
                    band_index,
                    common_valid,
                ] = values[common_valid]

            rr_output_path = (
                RR_DIR
                / f"Wald_Fused_Manda_{pair['output_label']}_10m.tif"
            )

            write_float_cube(
                rr_output_path,
                wald_fused,
                s2,
                nodata=NODATA,
            )

            baseline_rows, baseline_summary = evaluate_cube(
                reference_cube=reference,
                estimate_cube=baseline,
                spatial_reference_cube=planet_10m,
                valid_mask_array=common_valid,
                stage=stage,
                product="Baseline_Upsampled_30m_S2",
                resolution_ratio=RR_SCALE_FACTOR,
                hf_sigma=sigma_rr,
            )

            fused_rows, fused_summary = evaluate_cube(
                reference_cube=reference,
                estimate_cube=wald_fused,
                spatial_reference_cube=planet_10m,
                valid_mask_array=common_valid,
                stage=stage,
                product="Wald_HPF_Fused_10m",
                resolution_ratio=RR_SCALE_FACTOR,
                hf_sigma=sigma_rr,
            )

            wald_band_rows.extend(baseline_rows)
            wald_band_rows.extend(fused_rows)

            baseline_summary["output_path"] = ""
            fused_summary["output_path"] = str(
                rr_output_path
            )

            wald_summary_rows.append(
                baseline_summary
            )

            wald_summary_rows.append(
                fused_summary
            )

            print(
                "Valid pixels:",
                f"{int(common_valid.sum()):,}",
            )

            print(
                "Baseline mean RMSE:",
                round(
                    baseline_summary["mean_rmse"],
                    6,
                ),
            )

            print(
                "Fused mean RMSE:",
                round(
                    fused_summary["mean_rmse"],
                    6,
                ),
            )

wald_per_band_df = pd.DataFrame(
    wald_band_rows
)

wald_summary_df = pd.DataFrame(
    wald_summary_rows
)

display(wald_per_band_df.round(6))
display(wald_summary_df.round(6))

wald_per_band_df.to_csv(
    REPORT_DIR
    / "Q1_Wald_PerBand_Metrics.csv",
    index=False,
)

wald_summary_df.to_csv(
    REPORT_DIR
    / "Q1_Wald_Date_Product_Summary.csv",
    index=False,
)

print("✅ Wald reduced-resolution validation complete.")


In [ ]:
# CELL 12 — Final Q1 publication matrices

baseline_summary = (
    wald_summary_df[
        wald_summary_df["product"]
        == "Baseline_Upsampled_30m_S2"
    ]
    .set_index("stage")
)

fused_summary = (
    wald_summary_df[
        wald_summary_df["product"]
        == "Wald_HPF_Fused_10m"
    ]
    .set_index("stage")
)

publication_rows = []

for stage in ["Jan", "Mar", "Apr1"]:
    baseline_row = baseline_summary.loc[stage]
    fused_row = fused_summary.loc[stage]

    publication_rows.append({
        "Fusion_pair": stage,
        "RMSE": fused_row["mean_rmse"],
        "MAE": fused_row["mean_mae"],
        "PSNR_dB": fused_row["mean_psnr_db"],
        "SSIM": fused_row["mean_ssim"],
        "CC": fused_row["mean_cc"],
        "SAM_degrees": fused_row["sam_degrees"],
        "ERGAS": fused_row["ergas"],
        "UIQI": fused_row["mean_uiqi"],
        "Spatial_HF_CC": (
            fused_row["mean_spatial_hf_cc"]
        ),
        "Valid_pixels": int(
            fused_row["valid_pixels"]
        ),
        "Delta_RMSE_vs_baseline": (
            fused_row["mean_rmse"]
            - baseline_row["mean_rmse"]
        ),
        "Delta_PSNR_vs_baseline": (
            fused_row["mean_psnr_db"]
            - baseline_row["mean_psnr_db"]
        ),
        "Delta_SSIM_vs_baseline": (
            fused_row["mean_ssim"]
            - baseline_row["mean_ssim"]
        ),
        "Delta_CC_vs_baseline": (
            fused_row["mean_cc"]
            - baseline_row["mean_cc"]
        ),
        "Delta_SAM_vs_baseline": (
            fused_row["sam_degrees"]
            - baseline_row["sam_degrees"]
        ),
        "Delta_ERGAS_vs_baseline": (
            fused_row["ergas"]
            - baseline_row["ergas"]
        ),
        "Delta_UIQI_vs_baseline": (
            fused_row["mean_uiqi"]
            - baseline_row["mean_uiqi"]
        ),
        "Delta_HF_CC_vs_baseline": (
            fused_row["mean_spatial_hf_cc"]
            - baseline_row["mean_spatial_hf_cc"]
        ),
    })

publication_matrix = pd.DataFrame(
    publication_rows
)

display(publication_matrix.round(6))

publication_matrix.to_csv(
    REPORT_DIR
    / "Q1_Publication_Fusion_Metrics_Matrix.csv",
    index=False,
)

# Reviewer-friendly direction table.
metric_direction = pd.DataFrame(
    [
        ["RMSE", "Lower"],
        ["MAE", "Lower"],
        ["PSNR", "Higher"],
        ["SSIM", "Higher; closer to 1"],
        ["CC", "Higher; closer to 1"],
        ["SAM", "Lower"],
        ["ERGAS", "Lower"],
        ["UIQI", "Higher; closer to 1"],
        ["Spatial HF-CC", "Higher; closer to 1"],
    ],
    columns=[
        "Metric",
        "Preferred_direction",
    ],
)

metric_direction.to_csv(
    REPORT_DIR
    / "Q1_Metric_Interpretation.csv",
    index=False,
)

print("✅ Q1 publication matrix saved:")
print(
    REPORT_DIR
    / "Q1_Publication_Fusion_Metrics_Matrix.csv"
)


In [ ]:
# CELL 13 — Publication quicklook figures

def rgb_stretch(cube, valid):
    # Input band order: Blue, Green, Red, NIR
    rgb = cube[[2, 1, 0]].transpose(1, 2, 0)

    output = np.zeros_like(rgb, dtype="float32")

    for channel in range(3):
        values = rgb[:, :, channel][valid]

        if values.size == 0:
            continue

        low, high = np.percentile(
            values,
            [2, 98],
        )

        if high <= low:
            high = low + 1e-6

        output[:, :, channel] = np.clip(
            (rgb[:, :, channel] - low)
            / (high - low),
            0,
            1,
        )

    output[~valid] = 1.0
    return output


for stage, pair in PAIRS.items():
    rr_path = (
        RR_DIR
        / f"Wald_Fused_Manda_{pair['output_label']}_10m.tif"
    )

    with rasterio.open(pair["s2"]) as s2:
        reference = s2.read().astype("float32")

        with rasterio.open(rr_path) as rr:
            fused_rr = rr.read().astype("float32")

        reference_valid = valid_mask(
            reference,
            (
                s2.nodata
                if s2.nodata is not None
                else NODATA
            ),
        )

        fused_valid = valid_mask(
            fused_rr,
            NODATA,
        )

        valid = reference_valid & fused_valid

        figure = plt.figure(
            figsize=(12, 5)
        )

        axis1 = figure.add_subplot(1, 2, 1)
        axis1.imshow(
            rgb_stretch(reference, valid)
        )
        axis1.set_title(
            f"{stage}: Original Sentinel-2 10 m"
        )
        axis1.axis("off")

        axis2 = figure.add_subplot(1, 2, 2)
        axis2.imshow(
            rgb_stretch(fused_rr, valid)
        )
        axis2.set_title(
            f"{stage}: Wald HPF fused 10 m"
        )
        axis2.axis("off")

        figure.tight_layout()

        figure_path = (
            REPORT_DIR
            / f"Q1_Wald_Quicklook_{stage}.png"
        )

        figure.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(figure)

        print("Saved:", figure_path)

print("✅ Publication quicklooks complete.")


In [ ]:
# CELL 14 — Reproducibility manifest

manifest = {
    "study_area": "Manda Upazila, Naogaon, Bangladesh",
    "fusion_method": "High-Pass Filter (HPF) injection",
    "operational_extent": "Union Planet grid; no administrative AOI clip",
    "sentinel2_extent_rule": "Warped and clipped to Planet grid",
    "planet_dates": [
        "2026-01-25",
        "2026-03-05",
        "2026-04-07",
        "2026-04-22",
    ],
    "sentinel2_dates_used_for_fusion": [
        "2026-01-23",
        "2026-03-04",
        "2026-04-10",
    ],
    "full_resolution_validation": {
        "spectral_reference": (
            "Sentinel-2 resampled to Planet grid"
        ),
        "spatial_reference": (
            "Planet high-frequency information"
        ),
        "maximum_sample_pixels_per_date": 300000,
        "random_seed": RANDOM_SEED,
    },
    "wald_validation": {
        "reference_resolution_m": 10,
        "degraded_resolution_m": 30,
        "scale_factor": RR_SCALE_FACTOR,
        "reference": (
            "Original Sentinel-2 10 m"
        ),
        "baseline": (
            "30 m degraded Sentinel-2 "
            "upsampled to 10 m"
        ),
    },
    "metrics": [
        "RMSE",
        "MAE",
        "PSNR",
        "SSIM",
        "CC",
        "SAM",
        "ERGAS",
        "UIQI",
        "Spatial HF-CC",
    ],
    "nodata": NODATA,
    "hpf_gain": HPF_GAIN,
    "metric_data_range": METRIC_DATA_RANGE,
    "ssim_tile_size": SSIM_TILE_SIZE,
    "ssim_min_valid_fraction": (
        SSIM_MIN_VALID_FRACTION
    ),
}

manifest_path = (
    REPORT_DIR
    / "Q1_Fusion_Reproducibility_Manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )

print("✅ Reproducibility manifest:", manifest_path)
print()
print("FINAL OUTPUT FOLDERS")
print("Fused images:", FUSION_DIR)
print("Fused NDVI:", FUSION_NDVI_DIR)
print("Wald outputs:", RR_DIR)
print("Publication reports:", REPORT_DIR)
print("Classification inputs:", CLASSIFICATION_DIR)
